In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          TrainingArguments, Trainer, pipeline)
import evaluate

print("GPU available:", torch.cuda.is_available())

In [ ]:
raw = load_dataset("tner/ontonotes5")
print(raw)
print(raw["train"][0])

In [ ]:
onto_label2id = {"O":0,"B-CARDINAL":1,"B-DATE":2,"I-DATE":3,"B-PERSON":4,"I-PERSON":5,"B-NORP":6,"B-GPE":7,"I-GPE":8,
 "B-LAW":9,"I-LAW":10,"B-ORG":11,"I-ORG":12,"B-PERCENT":13,"I-PERCENT":14,"B-ORDINAL":15,"B-MONEY":16,"I-MONEY":17,
 "B-WORK_OF_ART":18,"I-WORK_OF_ART":19,"B-FAC":20,"B-TIME":21,"I-CARDINAL":22,"B-LOC":23,"B-QUANTITY":24,
 "I-QUANTITY":25,"I-NORP":26,"I-LOC":27,"B-PRODUCT":28,"I-TIME":29,"B-EVENT":30,"I-FAC":31,"B-LANGUAGE":32,
 "I-PRODUCT":33,"I-ORDINAL":34,"I-LANGUAGE":35,"I-EVENT":36}
onto_id2label = {v:k for k,v in onto_label2id.items()}

keep = {"PERSON":"PER","ORG":"ORG","GPE":"LOC","LOC":"LOC","DATE":"DATE","MONEY":"MONEY"}

label_list = ["O","B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC","B-DATE","I-DATE","B-MONEY","I-MONEY"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

def convert_tags(tags):
    out = []
    for t in tags:
        name = onto_id2label[t]
        if name == "O":
            out.append(label2id["O"]); continue
        prefix, ent = name.split("-", 1)
        out.append(label2id[f"{prefix}-{keep[ent]}"] if ent in keep else label2id["O"])
    return out

dataset = raw.map(lambda x: {"ner_tags": convert_tags(x["tags"])})

# Sanity check
ex = dataset["train"][0]
print(list(zip(ex["tokens"], [id2label[t] for t in ex["ner_tags"]])))

In [ ]:
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=128)
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev, ids = None, []
        for w in word_ids:
            if w is None: ids.append(-100)
            elif w != prev: ids.append(labels[w])
            else: ids.append(-100)
            prev = w
        all_labels.append(ids)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = dataset.map(tokenize_and_align, batched=True,
                           remove_columns=dataset["train"].column_names)

# Use a subset for faster training in the lab (remove .select() for full training)
train_ds = tokenized_ds["train"].shuffle(seed=42).select(range(8000))
val_ds   = tokenized_ds["validation"].select(range(1500))
test_ds  = tokenized_ds["test"].select(range(1500))
print(len(train_ds), len(val_ds), len(test_ds))

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds  = [[label_list[p] for p,l in zip(pr,lb) if l != -100] for pr,lb in zip(preds, labels)]
    true_labels = [[label_list[l] for p,l in zip(pr,lb) if l != -100] for pr,lb in zip(preds, labels)]
    r = seqeval.compute(predictions=true_preds, references=true_labels)
    return {"precision": r["overall_precision"], "recall": r["overall_recall"],
            "f1": r["overall_f1"], "accuracy": r["overall_accuracy"]}

In [ ]:
args = TrainingArguments(
    output_dir="bert-ner-output",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=data_collator, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
results = trainer.evaluate(test_ds)
print(results)

In [ ]:
trainer.save_model("bert-ner-final")
tokenizer.save_pretrained("bert-ner-final")

In [ ]:
ner = pipeline("token-classification", model="bert-ner-final", tokenizer="bert-ner-final",
               aggregation_strategy="simple",
               device=0 if torch.cuda.is_available() else -1)

def show(sentences):
    for s in sentences:
        print("\nSentence:", s)
        for e in ner(s):
            print(f"   {e['word']:<20} -> {e['entity_group']:<6} (score={e['score']:.2f})")

prelab = [
    "Sundar Pichai is the CEO of Google.",
    "Microsoft was founded by Bill Gates.",
    "The conference will be held in Chennai on 20 July 2026.",
    "Elon Musk visited India last week.",
]
show(prelab)

In [ ]:
postlab = [
    "Satya Nadella visited Hyderabad.",
    "Apple launched a new iPhone in California.",
]
show(postlab)

In [ ]:
show(["Amazon paid $1.2 billion for the company on 5 March 2021."])